# ChipFF Bench: thermal expansion on a free GPU (Kaggle or Colab)

This notebook runs the GPU part of ChipFF Bench: constant-pressure MD with kUPS for every material × model.
The CPU parts (lattice constants, elastic tensors) are already in the repo under `results/raw/`.

**Before you start**
* **Kaggle:** Settings → Accelerator → *GPU T4 x2* (or P100), and Settings → Internet → *On*.
  For UMA, add a secret named `HF_TOKEN` (Add-ons → Secrets) holding a Hugging Face token whose account accepted the UMA license.
* **Colab:** Runtime → Change runtime type → *T4 GPU*. For UMA, add `HF_TOKEN` under the key icon (Secrets).

Then *Run all*. Step 4 is a short pilot that prints timings; step 5 is the full run. When it finishes, download
`chipff_thermal_results.zip` and unzip it into the repo root on your computer (it only contains `results/raw/*/thermal*.json`).

## 1. Settings

In [ ]:
REPO = "https://github.com/YOUR-GITHUB-USERNAME/chipff-bench"   # <- your public repo
BRANCH = "main"
MODELS = ["mace-mpa-0", "orb-v3", "mattersim-5m", "uma-s-1p2"]
RUN_PILOT = True      # 2 models x 6 materials, 20 ps: checks the method and prints timings
RUN_FULL = True       # everything; results are saved per batch, so a re-run resumes where it stopped

## 2. Install (Python 3.12 via uv, since kUPS needs Python ≥ 3.12)

In [ ]:
import os, subprocess, sys
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install uv
if not os.path.exists("chipff-bench"):
    !git clone -q --branch {BRANCH} {REPO} chipff-bench
%cd chipff-bench
!uv venv -q --python 3.12 .venv
!uv pip install -q --python .venv/bin/python -e ".[uma,cuda]"
!.venv/bin/python -c "import jax; print('JAX devices:', jax.devices())"

## 3. Hugging Face token (only needed for UMA)

In [ ]:
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
    print("HF token found: UMA will run")
else:
    MODELS = [m for m in MODELS if not m.startswith("uma")]
    print("No HF_TOKEN secret: skipping UMA. Models:", MODELS)

## 4. Pilot: method check + timings

In [ ]:
!.venv/bin/chipff plan
if RUN_PILOT:
    !.venv/bin/chipff thermal -m mace-mpa-0 orb-v3 --materials Si Cu MgO GaN Al2O3 W --max-prod-ps 20 --tag thermal_pilot
    !.venv/bin/python scripts/thermal_status.py --tag thermal_pilot

## 5. Full run (resumable)
Each model runs separately and saves as it goes. If the session times out, just run this cell again.

In [ ]:
if RUN_FULL:
    for m in MODELS:
        !.venv/bin/chipff thermal -m {m}
    !.venv/bin/python scripts/thermal_status.py --tag thermal

## 6. Download the results

In [ ]:
!zip -q -r /kaggle/working/chipff_thermal_results.zip results/raw/*/thermal*.json 2>/dev/null || zip -q -r chipff_thermal_results.zip results/raw/*/thermal*.json
try:
    from google.colab import files; files.download("chipff_thermal_results.zip")
except Exception:
    print("Kaggle: find chipff_thermal_results.zip under Output (right panel) and download it.")